# 02 — Fine-tune FLAN-T5-base on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sissikhaled/medical-text-simplifier/blob/main/notebooks/02_train_colab.ipynb)

This notebook runs `src/train.py` from the GitHub repository on a Colab GPU. Everything that must survive a disconnect is written to **Google Drive**:

```
MyDrive/medical-text-simplifier/
    checkpoints/<run_name>/checkpoint-*/   per-epoch checkpoints (resumable, ~3 GB each)
    checkpoints/<run_name>/best/           best model (~1 GB), used for evaluation and the demo
    runs/<run_name>/                       config, loss CSV + plot, summary, validation samples
```

**How to use**
1. `Runtime → Change runtime type → T4 GPU`.
2. Pick the run in the **Settings** cell (one config per session).
3. `Runtime → Run all`. Keep the browser tab open; the free tier disconnects idle sessions.
4. If the session dies: reconnect and **Run all** again. It detects the existing checkpoint and resumes automatically (the early-stopping patience counter restarts from 0 after a resume).
5. At the end, download the zipped run folder (last cell) and send it back so it can be committed to the repository.

Expected time on a T4 (fp32, 1024-token inputs): very roughly 35–50 min per epoch; early stopping usually ends training before 10 epochs.

## 1. Settings

In [ ]:
# Which experiment to run (one per session):
#   configs/base.yaml    main run: lr 3e-4, inputs 1024 tokens
#   configs/lr1e-4.yaml  variation (a): lr 1e-4
#   configs/src512.yaml  variation (b): inputs 512 tokens
CONFIG = "configs/base.yaml"

# If training crashes with "CUDA out of memory", set this to True: it uses
# batch 2 x 8 accumulation steps instead of 4 x 4 (same effective batch of 16,
# so results stay comparable; just slower).
LOW_MEMORY = False

# A finished run is never retrained by accident. Set True to train it again
# from scratch (overwrites its run files; delete its old checkpoints first).
RETRAIN = False

REPO_URL = "https://github.com/sissikhaled/medical-text-simplifier.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/medical-text-simplifier"

## 2. GPU check
Must show a GPU (Tesla T4 or better). If it fails, change the runtime type and run again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Google Drive
Authorize access when asked. Checkpoints and run files are stored here.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
RUNS_DIR = f"{DRIVE_ROOT}/runs"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
!df -h /content/drive | tail -1

## 4. Code
Clone the repository (or update it if this session already cloned it).

In [ ]:
%cd /content
if os.path.isdir("medical-text-simplifier/.git"):
    !git -C medical-text-simplifier fetch -q origin {BRANCH} && git -C medical-text-simplifier reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {REPO_URL}
%cd /content/medical-text-simplifier
!git log --oneline -1

## 5. Packages
Installs the pinned versions from `requirements.txt` **except** the ones Colab already provides and that we do not need here: `torch` (Colab's build has CUDA), plus `pandas`, `matplotlib`, `jupyter` and `gradio` (replacing them can break Colab's own tools).

In [ ]:
!grep -v -E "^(torch|pandas|matplotlib|jupyter|gradio)==" requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt
!python -c "import torch, transformers, datasets; print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__, '| CUDA', torch.cuda.is_available())"

## 6. Train
Resumes automatically if this run already has a checkpoint on Drive.

In [ ]:
import glob, yaml
cfg = yaml.safe_load(open(CONFIG))
RUN_NAME = cfg["run_name"]
TRAIN_CONFIG = CONFIG
if LOW_MEMORY:
    cfg["per_device_train_batch_size"] = cfg["per_device_eval_batch_size"] = 2
    cfg["gradient_accumulation_steps"] = 8
    TRAIN_CONFIG = "/tmp/train_config_low_memory.yaml"
    yaml.safe_dump(cfg, open(TRAIN_CONFIG, "w"), sort_keys=False)

FINISHED = os.path.exists(f"{RUNS_DIR}/{RUN_NAME}/train_summary.json")
existing = glob.glob(f"{CKPT_DIR}/{RUN_NAME}/checkpoint-*")
RESUME = "--resume" if existing and not RETRAIN else ""
print(f"Run: {RUN_NAME}  (batch {cfg['per_device_train_batch_size']} x {cfg['gradient_accumulation_steps']})")
print(f"Existing checkpoints: {sorted(os.path.basename(p) for p in existing) or 'none'}")
if FINISHED and not RETRAIN:
    print("-> already finished: training will be skipped (set RETRAIN = True to train again)")
else:
    print("-> resuming" if RESUME else "-> starting from scratch")

In [ ]:
if FINISHED and not RETRAIN:
    print("Skipped: this run already finished.")
else:
    !python -m src.train --config {TRAIN_CONFIG} --output-dir "{CKPT_DIR}" --runs-dir "{RUNS_DIR}" {RESUME}

## 7. Results

In [ ]:
import json
from IPython.display import Image, display
run_dir = f"{RUNS_DIR}/{RUN_NAME}"
print(json.dumps(json.load(open(f"{run_dir}/train_summary.json")), indent=2))
display(Image(f"{run_dir}/loss_curve.png"))

In [ ]:
# First validation sample generated by the best model
with open(f"{run_dir}/val_samples.jsonl") as f:
    s = json.loads(f.readline())
print("SOURCE:\n", s["source"][:1200], "\n")
print("REFERENCE:\n", s["reference"][:1200], "\n")
print("MODEL:\n", s["prediction"][:1200])

## 8. Free Drive space (optional)
The per-epoch `checkpoint-*` folders (~3 GB each) are only needed to resume an unfinished run. Once training has **finished**, delete them and keep `best/` (~1 GB). The free Drive has 15 GB, so do this before starting the next run. Set `DELETE = True` to confirm.

In [ ]:
import shutil
DELETE = False
ckpts = sorted(glob.glob(f"{CKPT_DIR}/{RUN_NAME}/checkpoint-*"))
summary_exists = os.path.exists(f"{RUNS_DIR}/{RUN_NAME}/train_summary.json")
print("Checkpoints:", [os.path.basename(c) for c in ckpts])
print("Training finished (train_summary.json present):", summary_exists)
if DELETE and summary_exists:
    for c in ckpts:
        shutil.rmtree(c)
    print("Deleted. Kept:", os.listdir(f"{CKPT_DIR}/{RUN_NAME}"))
elif DELETE:
    print("Not deleted: training has not finished yet.")

## 9. Download the run files
Downloads `<run_name>.zip` (a few hundred KB: config, loss CSV and plot, summary, validation samples). Put the unzipped folder into `runs/` of the local repository so it can be committed. If the browser download is blocked, the same files are in `MyDrive/medical-text-simplifier/runs/<run_name>/`.

In [ ]:
from google.colab import files
zip_path = shutil.make_archive(f"/content/{RUN_NAME}", "zip", RUNS_DIR, RUN_NAME)
files.download(zip_path)